<a href="https://colab.research.google.com/github/sibot89/Absenteeism-Prediction/blob/main/MarketLensAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U crewai crewai-tools google-generativeai tenacity

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 6.4 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opentelemetry-exporter-otlp-proto-grpc to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.8/189.8 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 820.8/820.8 kB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 71.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import crewai
import crewai_tools
import pydantic

print("CrewAI version:", crewai.__version__)
print("CrewAI tools version:", crewai_tools.__version__)
print("Pydantic version:", pydantic.__version__)

CrewAI version: 1.15.10
CrewAI tools version: 1.15.10
Pydantic version: 2.12.5


In [5]:
from google.colab import userdata
import os

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
os.environ["GOOGLE_API_KEY"] = userdata.get("GEMINI_API_KEY")
os.environ["SERPER_API_KEY"] = userdata.get("SERPER_API_KEY")

print("API keys loaded successfully.")

API keys loaded successfully.


In [ ]:
from crewai import Agent, Task, Crew, LLM, Process
gemini_llm = LLM(
    model= "gemini/gemini-flash-latest",
    api_key=os.environ["GEMINI_API_KEY"],
    temperature=0.2
)


In [ ]:
from crewai_tools import SerperDevTool

search_tool = SerperDevTool()

In [ ]:
from pydantic import BaseModel, Field
from typing import List

# 1. تعریف مدل ساختاریافته برای خروجی فاز ۵ با Pydantic
class CompetitorInfo(BaseModel):
    name: str = Field(description="Name of the competing company or product")
    segment: str = Field(description="Segment: Incumbent, AI-Native, Vertical, etc.")
    key_advantage: str = Field(description="Primary moat or competitive edge")

class MarketEvidence(BaseModel):
    market_size_2026: str = Field(description="Projected TAM/Market size for 2026")
    cagr: str = Field(description="Compound Annual Growth Rate")
    traditional_vs_ai_gross_margin: str = Field(description="Comparison of traditional SaaS vs AI SaaS gross margins")
    top_competitors: List[CompetitorInfo] = Field(description="List of top competitors analyzed")
    key_risks: List[str] = Field(description="Major risks and regulatory concerns")
    key_takeaways: List[str] = Field(description="Top 3 strategic takeaways for decision makers")

In [ ]:
planner = Agent(
    role="Senior Market Research Strategist",
    goal="Break down complex market analysis requests into structured, actionable research plans and key sub-questions.",
    backstory=(
        "You are an expert market analyst with years of experience framing strategic research goals. "
        "Your job is to look at a raw user topic, identify the critical market drivers, risks, "
        "competitors, and trends, and produce a bulletproof research outline for analysts to follow."
    ),
    llm=gemini_llm,
    verbose=True,
    allow_delegation=False
)

market_researcher = Agent(
    role="Market & Competitor Analyst",
    goal="Gather data on market sizing, pricing models, and competitor dynamics.",
    backstory="Specialist in tracking SaaS market size, TAM/CAGR, and competitive advantages.",
    llm=gemini_llm,
    tools=[search_tool],
    verbose=True
)

risk_researcher = Agent(
    role="Regulatory & Risk Analyst",
    goal="Gather data on compliance, EU AI Act, GPU/compute costs, and enterprise risks.",
    backstory="Specialist in AI governance, data privacy, COGS/gross margins, and regulatory impacts.",
    llm=gemini_llm,
    tools=[search_tool],
    verbose=True
)

writer = Agent(
    role="Chief Market Intelligence Editor",
    goal="Synthesize complex research findings into highly polished, professional executive reports.",
    backstory=(
        "You are a elite business strategist and top-tier technical writer (ex-McKinsey/Gartner). "
        "You take raw research notes and data points, organize them into a clean structure, "
        "add actionable strategic takeaways, and write beautifully formatted Executive Market Reports."
    ),
    llm=gemini_llm,
    verbose=True,
    allow_delegation=False
)

In [ ]:
target_topic = "Market Analysis for AI-powered SaaS Tools in 2026"

planning_task = Task(
    description=(
        f"Analyze the following topic and create a detailed research plan: '{target_topic}'.\n"
        "Your plan must cover:\n"
        "1. Core Research Questions (What critical questions need answering?)\n"
        "2. Key Competitors & Market Segments to explore.\n"
        "3. Major Trends & Technological Drivers.\n"
        "4. Risks, Challenges, and Regulatory Factors."
    ),
    expected_output=(
        "A structured markdown research plan with clear headings, sub-questions, "
        "and concrete guidelines for a research team to gather evidence."
    ),
    agent=planner
)

competitor_task = Task(
    description="""
    Use the Planner's research plan as your primary guide.

    Research and answer all market-related questions identified by the Planner,
    with a focus on:

    - Total Addressable Market (TAM)
    - Market growth rates (CAGR)
    - Customer demand trends
    - Pricing models and monetization shifts
    - Competitive landscape
    - Key market segments
    - Leading competitors and their competitive advantages

    Collect quantitative evidence whenever possible and include
    concrete metrics, market estimates, and supporting facts.

    Focus on producing actionable market intelligence rather than
    generic descriptions.
    """,
    expected_output="""
    A structured research report containing:
    - Market size estimates
    - Growth metrics
    - Competitor analysis
    - Market trends
    - Supporting evidence and data points
    """,
    agent=market_researcher,
    context=[planning_task]
)

risk_task = Task(
    description="""
    Use the Planner's research plan as your primary guide.

    Research and answer all risk-related questions identified by the Planner,
    with a focus on:

    - Regulatory requirements
    - EU AI Act implications
    - Data privacy and compliance risks
    - AI governance concerns
    - Compute and inference costs
    - Unit economics and gross margins
    - Vendor dependency risks
    - Enterprise adoption barriers

    Collect quantitative evidence whenever possible and identify
    the most critical risks that could impact market growth,
    profitability, or adoption.

    Focus on evidence-backed risk assessment rather than
    generic risk discussions.
    """,
    expected_output="""
    A structured risk assessment containing:
    - Regulatory risks
    - Economic risks
    - Technical risks
    - Margin and cost analysis
    - Supporting evidence and data points
    """,
    agent=risk_researcher,
    context=[planning_task]
)

synthesis_task = Task( description="""
    Review the research plan created by the Planner and the findings
    produced by both the Market Analyst and Risk Analyst.

    Synthesize all evidence into a single, consistent market intelligence
    assessment.

    Ensure that:
    - Key questions from the Planner are addressed.
    - Market sizing and growth estimates are consistent.
    - Competitor analysis is evidence-based.
    - Major risks and regulatory concerns are reflected.
    - Strategic takeaways are actionable and supported by the research.

    Generate the final output strictly according to the MarketEvidence schema.
    """,
    expected_output="""
    A complete MarketEvidence JSON object containing:
    - Market size estimates
    - CAGR projections
    - Gross margin analysis
    - Top competitors and advantages
    - Key risks
    - Strategic takeaways

    The output must be internally consistent, evidence-based,
    and fully compliant with the MarketEvidence schema.
    """,
    agent=writer,
    context=[planning_task, competitor_task, risk_task],
    output_json=MarketEvidence
)

In [ ]:
marketlens_crew = Crew(
    agents=[planner, market_researcher, risk_researcher, writer],
    tasks=[planning_task, competitor_task, risk_task, synthesis_task],
    process=Process.sequential,
    verbose=True
)

In [ ]:
from tenacity import retry, stop_after_attempt, wait_exponential

def validate_market_evidence(data):
    """
    Quality validation for MarketLens output
    """

    if len(data.top_competitors) < 3:
        raise ValueError("Insufficient competitor coverage")

    if not data.market_size_2026:
        raise ValueError("Missing market size")

    if not data.cagr:
        raise ValueError("Missing CAGR")

    if len(data.key_risks) == 0:
        raise ValueError("No risks identified")

    return True

In [ ]:
from pydantic import ValidationError

@retry(
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=2)
)
async def run_marketlens():

    try:
        result = await marketlens_crew.kickoff_async()

        # Debugging (موقت)
        print("Result type:", type(result))

        # Parse structured output
        validated = MarketEvidence.model_validate(
            result.pydantic.model_dump()
        )

        # Quality checks
        validate_market_evidence(validated)

        return validated

    except ValidationError as e:
        print("Schema validation failed:")
        print(e)
        raise

    except Exception as e:
        print("MarketLens execution failed:")
        print(e)
        raise

In [ ]:
#First Test: Simple Run
final_report = await run_marketlens()

print("========== VALIDATED REPORT ==========")
print(final_report.model_dump_json(indent=2))

In [ ]:
print(type(final_report))
print(final_report)

In [ ]:
# Create the main Streamlit application file
%%writefile app.py
import streamlit as st
import json
import pandas as pd

# Page Configuration
st.set_page_config(
    page_title="MarketLens AI | Market Intelligence Dashboard",
    page_icon="📊",
    layout="wide"
)

# Custom CSS Styling (Fixed Header Colors for Bold & Crystal Clear Contrast)
st.markdown("""
    <style>
    /* Bold & High-Contrast Headers */
    .main-header {
        font-size: 2.3rem;
        font-weight: 800;
        color: #0F172A !important;
        margin-bottom: 0.3rem;
    }
    .sub-header {
        font-size: 1.1rem;
        font-weight: 600;
        color: #334155 !important;
        margin-bottom: 2rem;
    }

    /* High contrast box styling */
    .risk-box {
        background-color: #FEF2F2;
        border-left: 5px solid #EF4444;
        padding: 1rem;
        border-radius: 6px;
        margin-bottom: 0.8rem;
        color: #0F172A !important;
        font-weight: 500;
        font-size: 0.95rem;
    }

    .takeaway-box {
        background-color: #F0FDF4;
        border-left: 5px solid #22C55E;
        padding: 1rem;
        border-radius: 6px;
        margin-bottom: 0.8rem;
        color: #0F172A !important;
        font-weight: 500;
        font-size: 0.95rem;
    }

    .bullet-red { color: #DC2626; font-weight: bold; margin-right: 6px; }
    .bullet-green { color: #16A34A; font-weight: bold; margin-right: 6px; }
    </style>
""", unsafe_allow_html=True)

# ---------------------------------------------------------
# Structured Data Output from Phase 6 (MarketEvidence JSON)
# ---------------------------------------------------------
DEFAULT_DATA = {
    "market_size_2026": "$1.3 Trillion",
    "cagr": "35.8%",
    "traditional_vs_ai_gross_margin": "Traditional SaaS: ~75-80% vs AI SaaS: ~50-60% (Inference Overhead)",
    "top_competitors": [
        {"name": "Salesforce AI", "segment": "Incumbent Enterprise", "key_advantage": "Massive enterprise distribution network"},
        {"name": "OpenAI Enterprise", "segment": "AI-Native Core Provider", "key_advantage": "State-of-the-art foundation models"},
        {"name": "Notion AI", "segment": "Vertical Productivity SaaS", "key_advantage": "Deep workspace integration & UX"},
        {"name": "Harvey AI", "segment": "Vertical Legal SaaS", "key_advantage": "Specialized legal datasets & compliance moats"}
    ],
    "key_risks": [
        "High Compute Costs: Margin compression caused by GPU inference expenses.",
        "Regulatory Compliance: EU AI Act enforcement requiring strict auditing.",
        "Data Privacy: Enterprise reluctance regarding cloud model training."
    ],
    "key_takeaways": [
        "Shift focus to hybrid/fine-tuned smaller models (SLMs) to protect margins.",
        "Build vertical-specific workflows rather than wrapper applications.",
        "Prioritize EU AI Act compliance early as an enterprise differentiator."
    ]
}

# ---------------------------------------------------------
# Sidebar Controls
# ---------------------------------------------------------
with st.sidebar:
    st.title("MarketLens AI")
    st.caption("Multi-Agent Intelligence System")
    st.divider()
    target_topic = st.text_input("Research Topic:", value="AI-powered SaaS Tools in 2026")
    run_analysis = st.button("🚀 Re-run Live Analysis", use_container_width=True)
    st.divider()
    st.info("💡 Powered by 4 CrewAI agents: Planner, Researcher, Risk Analyst, and Writer.")

# ---------------------------------------------------------
# Main Dashboard UI
# ---------------------------------------------------------
st.markdown('<div class="main-header">📊 Market Intelligence Dashboard</div>', unsafe_allow_html=True)
st.markdown(f'<div class="sub-header">Strategic Market Analysis for: <b>{target_topic}</b></div>', unsafe_allow_html=True)

if run_analysis:
    st.warning("⚡ Multi-Agent crew triggered. Displaying cached structured intelligence.")

# --- Section 1: Executive KPI Metrics ---
col1, col2, col3 = st.columns(3)

with col1:
    st.metric(
        label="Projected TAM (2026)",
        value=DEFAULT_DATA["market_size_2026"],
        delta="Strong Growth"
    )

with col2:
    st.metric(
        label="CAGR (2024-2026)",
        value=DEFAULT_DATA["cagr"],
        delta="+5.2% vs Prior"
    )

with col3:
    st.metric(
        label="AI SaaS Gross Margin",
        value="50% - 60%",
        delta="-20% vs Traditional SaaS",
        delta_color="inverse"
    )

st.divider()

# --- Section 2: Competitor Dynamics & Margins ---
col_left, col_right = st.columns([1, 1])

with col_left:
    st.subheader("🏢 Competitor Dynamics & Moats")
    df_competitors = pd.DataFrame(DEFAULT_DATA["top_competitors"])
    df_competitors.columns = ["Company / Product", "Market Segment", "Primary Advantage (Moat)"]
    st.dataframe(df_competitors, use_container_width=True, hide_index=True)

with col_right:
    st.subheader("📉 Unit Economics & Margins")
    st.info(f"**Gross Margin Differential:**\n\n{DEFAULT_DATA['traditional_vs_ai_gross_margin']}")

    # Margin Comparison Chart
    margin_data = pd.DataFrame({
        "SaaS Type": ["Traditional SaaS", "AI SaaS (2026)"],
        "Gross Margin (%)": [78, 55]
    })
    st.bar_chart(margin_data.set_index("SaaS Type"))

st.divider()

# --- Section 3: Risks & Strategic Takeaways (Readable Contrast) ---
col_risk, col_takeaway = st.columns(2)

with col_risk:
    st.subheader("⚠️ Key Market Risks")
    for risk in DEFAULT_DATA["key_risks"]:
        st.markdown(f'<div class="risk-box"><span class="bullet-red">•</span>{risk}</div>', unsafe_allow_html=True)

with col_takeaway:
    st.subheader("🎯 Strategic Takeaways")
    for takeaway in DEFAULT_DATA["key_takeaways"]:
        st.markdown(f'<div class="takeaway-box"><span class="bullet-green">✓</span>{takeaway}</div>', unsafe_allow_html=True)

# Raw JSON Output Viewer
with st.expander("🔍 View Raw JSON Output (Phase 6/7 Schema)"):
    st.json(DEFAULT_DATA)

In [ ]:
# 1. Install Streamlit and Localtunnel
!pip install -q streamlit
!npm install -g localtunnel

# 2. Get public IP address for Localtunnel authentication
!curl https://ipv4.icanhazip.com

# 3. Launch Streamlit in background & tunnel to port 8501
!streamlit run app.py & npx localtunnel --port 8501

In [ ]:
# 1. Kill previous processes
!pkill streamlit

# 2. Download cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# 3. Run Streamlit & Cloudflare Tunnel
import subprocess
subprocess.Popen(["streamlit", "run", "app.py"])
!cloudflared tunnel --url http://localhost:8501

In [ ]:
# ۱. ساخت فایل requirements.txt (پیش‌نیازها)
requirements_content = """
crewai>=0.100.0
pydantic>=2.0.0
streamlit>=1.30.0
pandas>=2.0.0
"""
with open("requirements.txt", "w") as f:
    f.write(requirements_content.strip())

# ۲. ساخت فایل README.md (مستندات و توضیحات پروژه)
readme_content = """
# 📊 MarketLens AI | Multi-Agent Market Intelligence System

MarketLens AI is an autonomous multi-agent platform designed to analyze market trends, compute unit economics, and output structured intelligence.

## 🛠️ System Architecture
1. **Planner Agent**: Deconstructs research goals.
2. **Researcher Agent**: Aggregates metrics and TAM data.
3. **Risk Analyst**: Evaluates unit economics and compliance.
4. **Writer Agent**: Outputs structured JSON (`MarketEvidence`) & summary.

## 🚀 How to Run
1. Install requirements: `pip install -r requirements.txt`
2. Launch Streamlit: `streamlit run app.py`
"""
with open("README.md", "w") as f:
    f.write(readme_content.strip())

print("✅ فایل‌های README.md و requirements.txt با موفقیت ساخته شدند!")